20. 有效的括号

https://leetcode.cn/problems/valid-parentheses/description/?envType=study-plan-v2&envId=top-interview-150

给定一个只包括 '('，')'，'{'，'}'，'['，']' 的字符串 s，判断字符串是否有效。这题比较简单，一遍过。

In [4]:
class Solution:
    def isValid(self, s: str) -> bool:
        stack = []
        hashmap = {"(": ")", "{": "}", "[": "]"}
        for ch in s:
            if ch in hashmap:
                stack.append(ch)
            else:
                if not stack:
                    return False
                prev = stack.pop()
                prev_val = hashmap.get(prev, "")
                if prev_val != ch:
                    return False
        if stack:
            return False
        return True

71. 简化路径

https://leetcode.cn/problems/simplify-path/description/?envType=study-plan-v2&envId=top-interview-150

给你一个字符串 path ，表示指向某一文件或目录的 Unix 风格 绝对路径 （以 '/' 开头），请你将其转化为 更加简洁的规范路径。

这道题一开始要用 split 处理一下，就好做了，也就是你要想清楚栈的输入输出粒度，不要盲目就 char level。

In [15]:
path = "/a//b/.././c"
parts = path.split('/')
parts

['', 'a', '', 'b', '..', '.', 'c']

以下是完整代码。

In [ ]:
class Solution:
    def simplifyPath(self, path: str) -> str:
        stack = []
        parts = path.split('/')
        for part in parts:
            if part == "" or part == ".":
                continue
            elif part == "..":
                if stack:
                    stack.pop()
            else:
                stack.append(part)
        return "/" + "/".join(stack)

如下是 Shopee 春招的笔试题。

In [7]:
class Solution:
    def decodeStringWithPatterns(self, s):
        i = 0
        def parse(i):
            result = ""
            while i < len(s) and s[i] != "]":
                if s[i] == "#":
                    i += 2
                    op = ""
                    while s[i] != "}":
                        op += s[i]
                        i += 1
                    i += 2
                    content, i = parse(i)
                    if op == "upper":
                        result += content.upper()
                    elif op == "lower":
                        result += content.lower()
                    else:
                        result += content
                    i += 1
                elif s[i].isdigit():
                    k = 0
                    while s[i].isdigit():
                        k = k * 10 + int(s[i])
                        i += 1
                    i += 1
                    content, i = parse(i)
                    result += content * k
                    i += 1
                else:
                    result += s[i]
                    i += 1
            return result, i 
        content, _ = parse(0)    
        return content
    
solution = Solution()
solution.decodeStringWithPatterns(s="3[#{upper}[a]2[bc]]")

'AbcbcAbcbcAbcbc'

如下是我写的愚蠢至极的代码。

In [51]:
class Solution:
    def decodeStringWithPatterns(self, s):
        op_stack = []
        i = 0
        answer, text = "", ""
        while i < len(s):
            if s[i].isdigit():
                answer += text
                text = ""
                op = 0
                while s[i].isdigit():
                    op = op * 10 + int(s[i])
                    i += 1
                i += 1
                op_stack.append(op)      
            elif s[i] == "#":
                i += 2
                op = ""
                while s[i] != "}":
                    op += s[i]
                    i += 1
                op_stack.append(op)
                i += 2
            elif s[i] == "]":
                op = op_stack.pop()
                if type(op) == int:
                    if text:
                        answer += text * op
                    else:
                        answer = answer * op
                elif op == "upper":
                    answer = text.upper()
                else:
                    answer = text.lower()
                text = ""
                i += 1
            else:
                text += s[i]
                i += 1
        return answer
        
solution = Solution()
print(solution.decodeStringWithPatterns("3[a]2[bc]"))            # aaabcbc
print(solution.decodeStringWithPatterns("#{upper}[abc]"))         # ABC
print(solution.decodeStringWithPatterns("3[#{upper}[a]2[bc]]"))  # AbcbcAbcbcAbcbc
print(solution.decodeStringWithPatterns(s=r"ab2[c]")) # abcc
solution.decodeStringWithPatterns("#{upper}[abc2[a]]")

aaabcbc
ABC
AbcbcAbcbcAbcbc
abcc


''

如下是修正的代码。

In [49]:
def decodeStringWithPatterns(s: str) -> str:
    # 栈中每个元素保存：(已累积的字符串, 待执行的操作, 操作参数)
    # 操作类型: 'repeat' 或 'transform'
    stack = []
    cur = ""  # 当前层正在构建的字符串
    idx = 0

    while idx < len(s):
        ch = s[idx]

        if ch == '#':
            # 读取 #{modifier} 中的 modifier
            end = s.index('}', idx)
            modifier = s[idx+2 : end]  # 跳过 #{ 取到 }
            idx = end + 1  # 现在指向 [
            # 压栈：保存当前串、操作类型、参数
            stack.append((cur, 'transform', modifier))
            cur = ""

        elif ch.isdigit():
            # 读取完整数字（可能多位）
            j = idx
            while j < len(s) and s[j].isdigit():
                j += 1
            k = int(s[idx:j])
            idx = j  # 现在指向 [
            stack.append((cur, 'repeat', k))
            cur = ""

        elif ch == '[':
            # 数字/modifier 分支已经把 idx 对准了 [，这里统一跳过
            idx += 1
            continue

        elif ch == ']':
            # 弹出现场，把 cur 作为括号内的完整内容处理
            prev, op, param = stack.pop()
            if op == 'repeat':
                cur = prev + cur * param
            else:  # transform
                if param == 'upper':
                    cur = prev + cur.upper()
                elif param == 'lower':
                    cur = prev + cur.lower()
                else:
                    cur = prev + cur

        else:
            # 普通字母直接追加
            cur += ch

        idx += 1

    return cur

decodeStringWithPatterns(r"#{upper}[abc2[a]]")

'ABCAA'